# 04 · `[컴바인 + 경기 통합]` 결측·스케일링 방식 비교 & 누수 방지 파이프라인

이 노트북은 **컴바인 피처(T2) + 경기 타깃(T3·T4)을 통합**한 뒤, 모든 전처리 선택을 감이 아닌 **Train(408명) 5×5 층화 CV 숫자로 직접 비교·검증**한다.
- **절대 원칙**: 테스트셋(`test_ids.csv`, 102명)은 2절에서 격리한 뒤 **이 노트북 끝까지 절대 열지 않는다**.
- **비교 모델**: 이미 학습하신 `Ridge / Lasso / ElasticNet / PCA`만 사용해 전처리 효과를 투명하게 확인한다.

### [0] 실행 환경 부트스트랩 (로컬 / Colab 겸용)
- **데이터 범위**: `공통 환경`
- **작업 단계**: `환경 설정`
- **목적 및 원리**: `StopIteration` 없이 프로젝트 루트를 탐색하고 모듈을 불러온다.
- **확인할 내용**: `9/9 CSVs ready`.

In [ ]:
# =====================================================================
# [데이터: 공통] [단계: 환경 설정] 로컬/Colab 겸용 루트 탐색
# =====================================================================
import os, sys
from pathlib import Path

def find_project_root() -> Path:
    candidates = []
    if "NFL_PROJECT_ROOT" in os.environ:
        candidates.append(Path(os.environ["NFL_PROJECT_ROOT"]))
    candidates.extend([Path.cwd(), *Path.cwd().parents, Path("/content/nfl-bdb2027")])
    if "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        candidates.extend([
            Path("/content/drive/MyDrive/nfl-bdb2027"),
            Path("/content/drive/MyDrive/Colab Notebooks/nfl-bdb2027"),
        ])
    for p in candidates:
        if (p / "src" / "config.py").exists():
            return p.resolve()
    raise FileNotFoundError(f"src/config.py를 찾지 못했습니다 (cwd={Path.cwd()})")

ROOT = find_project_root()
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import pandas as pd
from src import config
from src.features import preprocessing as prep
config.setup()

### [1] 마스터 테이블 조인 (T2 + T3 + T4 → T5)
- **데이터 범위**: `컴바인+경기 통합`
- **작업 단계**: `데이터 통합`
- **목적 및 원리**: 510명 선수를 기준으로 컴바인 피처(T2)와 루키 경기 운동능력(T3)·역할/생산성(T4)을 Left Join한다.
- **확인할 내용**: 정확히 510행, 중복 `nfl_id` 0건.

In [ ]:
# =====================================================================
# [데이터: 컴바인+경기 통합] [단계: 데이터 통합] T5 마스터 테이블 생성
# =====================================================================
master = prep.build_master_table(save=True)
trials = pd.read_parquet(config.PROCESSED_FILES["combine_trials"])
display(master[["nfl_id", "display_name", "combine_position", "sub_position", "draft_year", "g_smax_top5mean", "g_smax_top5mean_cohort_z", "starter_flag"]].head(5))

### [2] 포지션 × 드래프트 티어 층화 80/20 분할 (테스트셋 봉인)
- **데이터 범위**: `컴바인+경기 통합`
- **작업 단계**: `데이터 분할 (누수 차단)`
- **목적 및 원리**: 모든 결측 대치·스케일링·Z-score 비교 전에 테스트셋 102명(`data/splits/test_ids.csv`)을 먼저 완전히 격리한다.
- **확인할 내용**: Train 408명 / Test 102명, 포지션·티어 비율 균일.

In [ ]:
# =====================================================================
# [데이터: 컴바인+경기 통합] [단계: 데이터 분할] 80/20 층화 고정 분할
# =====================================================================
train_df, test_df = prep.create_or_load_test_split(master, overwrite=False)
display(pd.crosstab(train_df["combine_position"], train_df["draft_tier"], margins=True))

### [3] 결측치 처리 방식 비교 (그룹 중앙값 vs KNN vs MICE)
- **데이터 범위**: `컴바인+경기 통합` (Train 408명 전용)
- **작업 단계**: `결측 방식 비교`
- **목적 및 원리**: 포지션 그룹 중앙값, 포지션 가중 KNN Imputer, 포지션 조건부 MICE(`IterativeImputer`)를 동일한 5×5 CV + Ridge로 공정 비교한다.
- **확인할 내용**: 전체 $R^2$뿐 아니라 **포지션 내부 잔차 $R^2$ (`r2_within_pos_mean`)** 비교.

In [ ]:
# =====================================================================
# [데이터: 컴바인+경기 통합] [단계: 결측 방식 비교] Group Median vs KNN vs MICE
# =====================================================================
imp_tbl = prep.compare_imputers_cv(train_df, trials, target_col="g_smax_top5mean")
display(imp_tbl)

### [4] 스케일링 · 분포 변환 방식 비교 (`Standard` vs `Robust` vs `Yeo-Johnson`)
- **데이터 범위**: `컴바인+경기 통합` (Train 408명 전용)
- **작업 단계**: `스케일링 비교`
- **목적 및 원리**: 이상치에 민감한 `StandardScaler`, 중앙값·IQR 기반 `RobustScaler`, 치우침을 펴주는 `Yeo-Johnson` 변환의 CV 성능을 비교한다.
- **확인할 내용**: 3가지 스케일러의 포지션 내부 $R^2$ 및 MAE 차이.

In [ ]:
# =====================================================================
# [데이터: 컴바인+경기 통합] [단계: 스케일링 비교] Standard vs Robust vs Yeo-Johnson
# =====================================================================
sc_tbl = prep.compare_scalers_cv(train_df, trials, target_col="g_smax_top5mean")
display(sc_tbl)

### [5] 피처 세트 점진적 효과 비교 (절대 타깃 & 시즌 코호트 상대 타깃)
- **데이터 범위**: `컴바인+경기 통합` (Train 408명 전용)
- **작업 단계**: `피처 추가 효과 검증`
- **목적 및 원리**: `FS-Trad`(전통 12개) → `FS-Phys`(+체격비율·물리·나이·결측플래그) → `FS-Biomech`(+3축 트래킹·40야드 F-V·스킬 드릴 구제·드릴별 Z-score) → `FS-Full`(+NGS)를 **절대 타깃과 시즌 코호트 상대 타깃** 양쪽에서 비교한다.
- **확인할 내용**: 생체역학·구제 피처 추가 시 포지션 내 $R^2$ 향상 폭.

In [ ]:
# =====================================================================
# [데이터: 컴바인+경기 통합] [단계: 피처 효과 검증] FS-Trad -> FS-Full
# =====================================================================
fs_tbl = prep.compare_feature_sets_cv(train_df, trials)
display(fs_tbl)

### [6] 차원 · 피처 생존 점검 (PCA 설명분산 & Lasso / ElasticNet 계수)
- **데이터 범위**: `컴바인+경기 통합` (Train 408명 전용)
- **작업 단계**: `차원·피처 점검` + `인코딩 검증`
- **목적 및 원리**: 이미 학습한 PCA로 유효 차원 수(80%·90% 분산)를 확인하고, Lasso/ElasticNet에서 어떤 피처(특히 포기 선수 구제용 스킬 드릴·물리 파생)가 살아남는지 확인한다.
- **확인할 내용**: 상위 20개 생존 피처 및 계수 부호의 물리적 타당성.

In [ ]:
# =====================================================================
# [데이터: 컴바인+경기 통합] [단계: 차원·피처 점검] PCA + Lasso/ElasticNet
# =====================================================================
pca_tbl, coef_tbl = prep.inspect_pca_and_lasso(train_df, trials, target_col="g_smax_top5mean_cohort_z")
display(pca_tbl.head(15))
display(coef_tbl.head(20))